> Nous remercions l’infrastructure de recherche IR* Huma-Num (n° ROR : 04ces3204 - n° RNSR : 201320910B - France) pour la mise à disposition des ressources informatiques nécessaires à ce travail.

# Après la retranscription avec les outils d'Huma-Num

## Transformer les fichiers ".srt" en ."csv"

Quand on utilise la fonction `with speaker` proposer par l'outil de retranscription d'Huma-Num Tools, le résultat nous est transmis au format `.txt` et `.srt`, mais au moment où j'écris ces lignes (le 06 mai 2026) il n'y a pas de format tabulé (`.tsv`). Or, il peut parfois être pratique de travailler les entretiens dans un tableur, lorsque l'on veut "classer" des extraits, appliquer une grille d'analyse.

Les scripts qui suivent ont donc pour objectif d'organiser les éléments contenues dans le fichier `.srt` dans un tableau (dataframe) et de les enregistrer dans un fichier `.csv`. Le résultat prendra la forme d'une fonction.
On prend le cas du fichier `.srt`, car il contient le minutage des segments, des informations utiles pour se repérer dans l'enregistrement et réécouter le passage si besoin.


## Ouvrir et lire le fichier ".srt"

Pour commencer, nous allons ouvrir, lire et traiter les "fichiers ".srt" (format de fichier utilisé pour les sous-titres sur youtube par exemple). 


In [6]:
gb_transcript = "../data/gb/transcription_originale/grazia_borrini_07-06-18.srt" # comme on va utiliser plusieur fois le fichier, on enregistre sa localisation dans une variable


'../data/gb/transcription_originale/grazia_borrini_07-06-18.srt'

In [11]:

with open(gb_transcript, 'r', encoding='utf-8-sig') as srt : #signifie qu'on ouvre le fichier et on le conserve dans l'objet srt
    lines = srt.readlines() #On lit chaque ligne du fichier et on les met dans une liste nommé lines
    for l in lines[0:20]: #On affiche les 20 premières lignes du fichier
        print(l)

1

00:00:01,820 --> 00:00:09,259

Speaker 1: Je suis arrivé en post-doctorat au Muséum National d'Historie Naturelle l'année dernière, en septembre.



2

00:00:09,299 --> 00:00:19,930

Speaker 1: C'est un projet porté par à la fois Stéphanie Duvaille et les écologues qui ont mis au point ce qu'on appelle les sciences participatives ou en anglais les citizen science.



3

00:00:20,441 --> 00:00:22,143

Speaker 0: Oh, Citizen Science, ok.



4

00:00:22,183 --> 00:00:32,453

Speaker 1: Avec l'idée de faire participer le public à la récolte de données, etc.



5

00:00:32,493 --> 00:00:35,517

Speaker 0: Peut-être direction de recherche aussi, n'est-ce pas?





In [3]:
mr_transcript = "" # comme on va utiliser plusieur fois le fichier, on enregistre sa localisation dans une variable

1

00:00:05,980 --> 00:00:15,620

Speaker 1: Donc je vais vous présenter un peu, redire un peu pourquoi on se rencontre et puis on intègre pas l'entretien.



2

00:00:15,660 --> 00:00:24,802

Speaker 1: Donc moi je suis Aymeric et j'ai été recruté en post-doctorat pour travailler sur la question des observatoires participatifs.



3

00:00:24,842 --> 00:00:25,542

Speaker 0: Et vous êtes.



4

00:00:25,582 --> 00:00:26,823

Speaker 0: quoi comme formation de doctorat ?



5

00:00:26,823 --> 00:00:27,943

Speaker 1: Moi je suis sociologue.



6

00:00:27,983 --> 00:00:29,843

Speaker 1: Je suis formé en sociologie.



7

00:00:29,863 --> 00:00:30,023

Speaker 1: D'accord.



8

00:00:30,043 --> 00:00:31,664

Speaker 1: J'ai fait une thèse en sociologie.



9

00:00:31,724 --> 00:00:33,666

Speaker 0: En sociologie sur l'environnement ?



10

00:00:33,666 --> 00:00:39,290

Speaker 1: Oui, j'ai travaillé sur les conflits environnementaux dans lesquels s'engageaient les riverains.





## Création du csv

On a besoin des trois modules ci-dessous

In [16]:
import time
import csv
import datetime


### Conversion du minutage en second

Pour la prochaine étape, nous allons avons besoin que les bornes temporelles des extraits soient exprimée en secondes. C'est ce que permet de faire la **fonction** `convert_to_second` (la fonction `convertir` fait l'inverse, elle convertit des millisecondes au format heures:minutes:secondes,millisecondes).




In [21]:
#(inspiration Yacine Chitour)
def convertir(seconds):
    """
    Convertisseur de secondes au format Heure:minutes:secondes,millisecondes)

    seconds : integer. Time in seconds

    Returns a string
    """
    
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms}"

def convert_to_second(str_time):
    """
    Convertit en seconde le temps exprimé au format Heure:minutes:secondes,millisecondes)

    str_time : string. Time au format HH:MM:SS,ms

    Returns an integer
    """
    h = int(str_time.split(":")[0])*3600
    m = int(str_time.split(":")[1])*60
    s= int(str_time.split(":")[2].split(",")[0])
    if "," in str_time:
        ms = float("0."+str_time.split(',')[-1])
    else:
        ms =0.0
    return h+m+s+ms
    

Sous Python, pour utiliser une fonction, il suffit d'écrire le nom de la fonction et de préciser entre parenthèse les arguments demandés par la fonction.
Par exemple, si je veux convertir 10 secondes au format "heures:minutes:secondes", j'écris `convertir(seconds=10)`

Quand on utilise une fonction pour la première fois, il est toujours utile de connaître les arguments que prend la fonction et ce qu'elle fait avec ces arguments. Pour cela, on peut aller voir la documentation disponible sur la page du module dont elle est issue ou exécuter les commandes `shift + tab`. À condition que la fonction est documentée...

In [19]:
time = input("Entrez une heure au format HH:MM:SS,ms : ")
hms = convert_to_second(time)

print('le temps exprimé en "HH:MM:SS,ms" : ', time)
print('le temps exprimé en millisecondes : ', hms)

le temps exprimé en "HH:MM:SS,ms" :  12:22:00
le temps exprimé en millisecondes :  44520.0


### Définition d'une fonction pour créer le .csv

La fonction `srt_to_csv` transforme le fichier srt au format csv. Elle prend en entrée l'adresse du fichier srt que l'on veut convertir. La fonction retourne en sortie un fichier csv avec comme colonne :

* l'identifiant du segment retranscrit ("id")
* les bornes de début et de fin du segment au format "HH:MM:SS,ms" ("start" et "end")
* les bornes de début et de fin du segment en millisecondes ("start_in_ms" et "end_in_ms")
* le locuteur identifié ("speaker")
* le texte ("text")


In [33]:
def srt_to_csv(path_of_file):
    """
    Convert srt file into csv.
    ------
    path_of_file : str. Chemin d'accès au fichier srt à convertir

    Return a csv file.
    """
    
    with open(path_of_file.replace('.srt', '.csv'), 'w', newline='') as csvfile:
        fieldnames = ['id', 'start', 'end','speaker', 'text','start_in_ms','end_in_ms']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        with open(path_of_file, 'r', encoding='utf-8-sig') as srt :
            i = 0
            lines = srt.readlines()
            dict_segment = {}
            list_segment = []
            for line in lines:
                i+=1
                #print("ligne : ", i)
                #print(line)
                if i ==4: #dans le fichier srt chaque segment "occupe" 4 ligne : (1)le numéri du segment, (2) l'horodatage, (3) texte, (4) le texteQuand on arrive à la quatrième ligne on écrit donc le resultat
                    writer.writerow({'id': 'seg_' + dict_segment['id'],
                                     'start': dict_segment['start'],
                                     'end': dict_segment['end'],
                                     'speaker': dict_segment['speaker'],
                                     'text':dict_segment['text'],
                                     'start_in_ms':dict_segment['start_in_ms'],
                                     'end_in_ms':dict_segment['end_in_ms']})
                    list_segment.append(dict_segment)
                    dict_segment =  {}
                    i = 0
                elif i == 1:
                    dict_segment["id"] = line.strip()
                elif i == 2:
                    dict_segment["start"] = line.split(" --> ")[0].strip()
                    dict_segment["start_in_ms"] = convert_to_second(dict_segment["start"])
                    dict_segment["end"] = line.split(" --> ")[1].strip()
                    dict_segment["end_in_ms"] = convert_to_second(dict_segment["end"])
                elif i == 3 :
                    dict_segment["speaker"] = line.split(": ")[0]
                    dict_segment["text"] = line.split(": ")[1].strip()


### Exercice :

Créez une cellule au format "code" et convertissez au format csv les fichiers srt correspondant aux retranscriptions des entretiens de Marie Roué et Grazia Borrini

In [34]:
srt_to_csv(path_of_file= gb_transcript)

**Fin de la première partie**